# 05 — Model family & feature set (team 07)

**Owner:** Patrick, filled in **after** 01–04 are merged into `team-07-main`, from their findings tables.
*(Filled in by Kingston for Deliverable 4.)*

This is the actual deliverable: a named model family, a feature set produced by code, and a reason
for every feature kept or dropped. Deliverable 4 builds directly on `X` and `y` from here.

Choose the model in the D2 order: **data shape → loss → what AA needs → cost → only then score.**

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR

## 1. Model family

**Chosen family:** gradient-boosted trees (XGBoost), trained as one model across all routes.

We chose XGBoost, a gradient-boosted tree model, trained as one model across all routes. Each route is its own monthly time series, but there are about 17,000 of them with only 91 months each, so fitting a separate ARIMA model per route would be slow and each one would see very little data. One shared tree model can learn patterns that repeat across routes, like summer peaks or small routes being more jumpy, and trees handle the wide range of route sizes, the airport codes, and missing months without extra work.

The other choice that shapes everything is how we set up the forecast. Deliverable 5 asks for a forecast of two years ahead as a whole, not just next month. So every row is built from a forecast date: the model may only use data up to that date, and it predicts each of the next 1 to 24 months, with the number of months ahead (h) as a feature. The test is one real forecast made at July 2024 for August 2024 through July 2026. Training uses forecast dates from July 2022 to June 2024, because before mid-2022 the history the features look back on is mostly COVID.

| EDA observation (from notebook #) | Points toward / away from |
|---|---|
| One row is one route in one month, about 17,000 routes, 91 months (01) | Toward one global model across all routes. Away from fitting 17,000 separate ARIMA models |
| Strong yearly seasonality, summer peaks and winter lows (03) | Toward month features and same-month-last-year features. Any model works if given these |
| Passengers range from under 1 to over 100,000 per route-month, heavy right tail (02, 03) | Toward training on log(1 + passengers). Trees handle the wide range well |
| COVID collapse in 2020 and recovery through 2022 (01, 03) | Away from models that need long clean history like LSTM. Toward training only on forecast dates after mid-2022 |
| 150 airports, 17,119 routes, airport table joins cleanly (04) | Toward trees, which can use airport codes as categories without one-hot blowing up |
| OD_REV_USD is known only after the month happens (03, 04) | Never use same-month revenue. Only past revenue (yield) is allowed |
| D5 needs a forecast two years ahead as a whole (DELIVERABLES.md) | Toward a forecast-date setup where every feature comes from before the forecast date |

## 2. Build the feature set

Shared settings for every Deliverable 4 model. Every candidate model should use the same forecast
dates, the same test window, and the same `score()` function so the results can be compared.

In [ ]:
import warnings

HORIZON = 24                                     # months ahead we forecast (D5: two years)
TEST_ORIGIN = pd.Timestamp("2024-07-01")         # last month we are allowed to see for the test forecast
FIRST_TRAIN_ORIGIN = pd.Timestamp("2022-07-01")  # first forecast date with a mostly post-COVID year of history
VAL_START = pd.Timestamp("2023-07-01")           # for early stopping: fit on targets before, validate on targets after

In [ ]:
def _row_mean(a):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN rows -> NaN is fine
        return np.nanmean(a, axis=1)


def _decimal_coord(a, kind):
    # AIRPORT_INFO stores degrees / minutes / seconds + hemisphere; turn it into one signed number
    deg = a[f"{kind}_DEGREE_QTY"] + a[f"{kind}_MINUTE_QTY"] / 60 + a[f"{kind}_SECOND_QTY"] / 3600
    return deg.where(~a[f"{kind}_HEMSPHR_CD"].isin(["S", "W"]), -deg)


def build_features(pax, airports, origins, horizon=HORIZON, max_target=None):
    """One row per (route, forecast date, target month).

    A forecast date (origin) is the last month we are allowed to see. For each origin we forecast the
    next 1..horizon months. Every feature is computed from months up to the origin, never after it,
    so the same code is leakage-safe for training and for the real two-year test forecast.
    """
    df = pax.copy()
    df["DATE"] = pd.to_datetime(dict(year=df["DEP_YEAR"], month=df["DEP_MONTH"], day=1))
    df["ROUTE"] = df["ORG_AIRPRT"] + "-" + df["DES_AIRPRT"]
    months = pd.date_range(df["DATE"].min(), df["DATE"].max(), freq="MS")
    col = {m: i for i, m in enumerate(months)}

    # route x month tables; a missing route-month stays NaN (01: missing means no data, not zero)
    wide_pax = df.pivot(index="ROUTE", columns="DATE", values=TARGET).reindex(columns=months)
    wide_rev = df.pivot(index="ROUTE", columns="DATE", values="OD_REV_USD").reindex(columns=months)
    routes = wide_pax.index
    P = wide_pax.to_numpy()
    YLD = (wide_rev / wide_pax).to_numpy()  # revenue per passenger; only ever used from past months
    org = routes.str.split("-").str[0].to_numpy()
    des = routes.str.split("-").str[1].to_numpy()
    org_tot = pd.DataFrame(P).groupby(org).sum(min_count=1).loc[org].to_numpy()  # whole-airport traffic
    des_tot = pd.DataFrame(P).groupby(des).sum(min_count=1).loc[des].to_numpy()
    pax_2019 = P[:, [col[m] for m in pd.date_range("2019-01-01", "2019-12-01", freq="MS")]]

    # static route features from AIRPORT_INFO (04: joins cleanly on AIRPRT_CD)
    a = airports.dropna(subset=["AIRPRT_CD"]).drop_duplicates("AIRPRT_CD").set_index("AIRPRT_CD")
    lat, lon = _decimal_coord(a, "LAT"), _decimal_coord(a, "LNGTD")
    lat1, lon1, lat2, lon2 = (np.radians(s.loc[k].to_numpy(dtype=float)) for s, k in
                              ((lat, org), (lon, org), (lat, des), (lon, des)))
    hav = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    distance_mi = 3958.8 * 2 * np.arcsin(np.sqrt(hav))
    domestic = ((a["CNTRY_CD"].loc[org].to_numpy() == "US") & (a["CNTRY_CD"].loc[des].to_numpy() == "US")).astype(int)
    airport_codes = sorted(set(org) | set(des))

    parts = []
    for origin in origins:
        o = col[pd.Timestamp(origin)]
        level_12 = _row_mean(P[:, o - 11:o + 1])
        level_3 = _row_mean(P[:, o - 2:o + 1])
        level_12_prev = _row_mean(P[:, o - 23:o - 11]) if o >= 23 else np.full(len(P), np.nan)
        for h in range(1, horizon + 1):
            t = o + h
            if t >= len(months) or (max_target is not None and months[t] > max_target):
                break
            month = months[t].month
            last_same = P[:, t - 12 * int(np.ceil(h / 12))]  # same calendar month, latest one at or before the origin
            parts.append(pd.DataFrame({
                "ROUTE": routes, "ORIGIN": months[o], "DATE": months[t], "h": h,
                "month_sin": np.sin(2 * np.pi * month / 12), "month_cos": np.cos(2 * np.pi * month / 12),
                "last_same_month": last_same,
                "level_12": level_12,
                "level_3": level_3,
                "same_vs_level12": np.log1p(last_same) - np.log1p(level_12),
                "level3_vs_level12": np.log1p(level_3) - np.log1p(level_12),
                "growth_12": np.log1p(level_12) - np.log1p(level_12_prev),
                "pax_2019_same_month": pax_2019[:, month - 1],
                "level12_vs_2019": np.log1p(level_12) - np.log1p(_row_mean(pax_2019)),
                "yield_12": _row_mean(YLD[:, o - 11:o + 1]),
                "org_level_12": _row_mean(org_tot[:, o - 11:o + 1]),
                "des_level_12": _row_mean(des_tot[:, o - 11:o + 1]),
                "months_seen_12": np.isfinite(P[:, o - 11:o + 1]).sum(axis=1),
                "distance_mi": distance_mi,
                "domestic": domestic,
                "ORG_AIRPRT": pd.Categorical(org, categories=airport_codes),
                "DES_AIRPRT": pd.Categorical(des, categories=airport_codes),
                TARGET: P[:, t],
            }))
    out = pd.concat(parts, ignore_index=True)
    return out[out[TARGET].notna()].reset_index(drop=True)  # only score months that actually exist


FEATURES = [
    "h", "month_sin", "month_cos",
    "last_same_month", "level_12", "level_3", "same_vs_level12", "level3_vs_level12", "growth_12",
    "pax_2019_same_month", "level12_vs_2019", "yield_12",
    "org_level_12", "des_level_12", "months_seen_12",
    "distance_mi", "domestic", "ORG_AIRPRT", "DES_AIRPRT",
]

train_origins = pd.date_range(FIRST_TRAIN_ORIGIN, TEST_ORIGIN - pd.DateOffset(months=1), freq="MS")
train_set = build_features(pax, airports, train_origins, max_target=TEST_ORIGIN)
test_set = build_features(pax, airports, [TEST_ORIGIN])

X, y = train_set[FEATURES], train_set[TARGET]
X_test, y_test = test_set[FEATURES], test_set[TARGET]
print(f"train: {len(train_set):,} rows from {len(train_origins)} forecast dates, "
      f"targets {train_set['DATE'].min():%Y-%m} to {train_set['DATE'].max():%Y-%m}")
print(f"test:  {len(test_set):,} rows, one forecast made at {TEST_ORIGIN:%Y-%m} for "
      f"{test_set['DATE'].min():%Y-%m} to {test_set['DATE'].max():%Y-%m}")

### Shared metrics and baseline

Every D4 candidate reports these on `test_set`. WAPE (total absolute error divided by total
passengers) is the headline number because it weights big routes more, like the business does.

In [ ]:
def score(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    err = predicted - actual
    return {
        "WAPE": np.abs(err).sum() / actual.sum(),   # share of passengers we got wrong
        "MAE": np.abs(err).mean(),                  # average miss per route-month, in passengers
        "RMSE": np.sqrt((err ** 2).mean()),         # punishes big misses on big routes
        "bias": err.sum() / actual.sum(),           # + means we forecast too many passengers overall
    }


def naive_forecast(df):
    """Seasonal naive: same month, latest year we've seen. Fallbacks for routes missing that month."""
    return (df["last_same_month"].fillna(df["level_12"]).fillna(df["pax_2019_same_month"])
            .fillna(df["level_12"].median()))


pd.DataFrame({"seasonal naive": score(y_test, naive_forecast(test_set))}).T.round(3)

## 3. Rationale per feature

| Feature | Kept / engineered / dropped | Reason (cite the EDA notebook) |
|---|---|---|
| `h` | Engineered | Months between the forecast date and the target month. Lets one model learn that far-out months are less certain (D5 two-year horizon) |
| `month_sin`, `month_cos` | Engineered | Target month as a circle so December sits next to January (03) |
| `last_same_month` | Engineered | Same month in the latest year we've seen. Captures each route's seasonality and size (03). It is also the naive baseline |
| `level_12`, `level_3` | Engineered | Route's average over the last 12 and last 3 months before the forecast date, its current size (01, 03) |
| `same_vs_level12`, `level3_vs_level12` | Engineered | How peaky the target month is for this route, and whether the route is trending up or down lately (03) |
| `growth_12` | Engineered | Last 12 months vs the 12 before, the route's yearly growth (03) |
| `pax_2019_same_month`, `level12_vs_2019` | Engineered | Pre-COVID level of the route, and how far it has recovered. 2019 is always before any forecast date (01, 03) |
| `yield_12` | Engineered | Average revenue per passenger over the last 12 months. Past revenue is allowed, same-month revenue is not (03, 04) |
| `org_level_12`, `des_level_12` | Engineered | Total traffic at each airport, a hub-size signal for small routes (04) |
| `months_seen_12` | Engineered | How many of the last 12 months the route reported. Routes with gaps behave differently (01) |
| `distance_mi` | Engineered | Great-circle distance from the airport latitudes and longitudes (04) |
| `domestic` | Engineered | Both airports in the US (04) |
| `ORG_AIRPRT`, `DES_AIRPRT` | Kept | 150 airports each, used as native categories in XGBoost instead of one-hot (04) |
| `DEP_YEAR` | Dropped | Trees can't extrapolate a year number into 2025 and 2026, and the trend is carried by the level features instead (03) |
| `OD_REV_USD` (same month) | Dropped | Leakage, only known after the month happens (03, 04) |
| `Unnamed: 0`, `SF_LOAD_TMS` | Dropped | Row counter and a single load timestamp (01) |
| `STATE_PROVNC_CD`, airport names | Dropped | 52% missing, and names add nothing beyond the code (01, 04) |
| Route as one category (17,119 values) | Dropped | Too many levels. Origin plus destination plus the route's own history carry the same information (04) |

Every feature above is computed only from months up to the forecast date, so none of them can leak the answer. The model is trained on log(1 + passengers) because route sizes range from under 1 to over 100,000 passengers a month, and the test forecasts are turned back into passengers before scoring. Missing route-months are left out of scoring and show up as gaps in the features, which XGBoost handles on its own.